<a href="https://colab.research.google.com/github/somendrew/Fine-tuning_LLMs/blob/main/QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## QLoRA in LLMs (Quantized LoRA)

**QLoRA (Quantized Low-Rank Adaptation)** is a parameter-efficient fine-tuning method that allows you to fine-tune **very large language models** using **low GPU memory** by combining:

* **Quantization** → compress base model weights (usually 4-bit)
* **LoRA adapters** → train only small additional matrices instead of the full model

This makes it possible to fine-tune 7B–70B parameter models on a single GPU.

---

## 1. Problem QLoRA solves

Traditional fine-tuning:

* Requires updating **all model parameters**
* Needs huge GPU memory
* Expensive and slow

**LoRA** solved this partly by training small adapter matrices, but the **base model still remained in full precision**, consuming large memory.

**QLoRA improvement:**

* Base model → **4-bit quantized (frozen)**
* Only **LoRA adapters (small matrices)** are trained in higher precision

So memory usage drops dramatically.

---

## 2. How QLoRA Works (Concept)

Workflow:

1. Load pretrained LLM
2. Quantize model weights to **4-bit (NF4 quantization)**
3. Freeze base model
4. Insert **LoRA adapters** into attention layers
5. Train only LoRA parameters

Mathematically:

Instead of updating weight ( W ):

[
W' = W + BA
]

Where:

* (W) = frozen quantized weight
* (A, B) = small trainable low-rank matrices

Only **A and B** are trained.

---

## 3. Key Innovations of QLoRA

QLoRA introduced three important techniques:

1. **NF4 (NormalFloat4)**
   Special 4-bit datatype optimized for normally distributed weights.

2. **Double Quantization**
   Further compress quantization constants → reduces memory more.

3. **Paged Optimizers**
   Prevent GPU memory spikes during training.

These enable fine-tuning huge models even on limited hardware.

---

## 4. Memory Comparison (Example)

Example: 13B model

| Method           | Memory Needed |
| ---------------- | ------------- |
| Full fine-tuning | ~80-100 GB    |
| LoRA (fp16)      | ~30-40 GB     |
| **QLoRA**        | ~10-15 GB     |

Huge difference.

---

## 5. Simple QLoRA Code (HuggingFace)

Basic workflow:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    load_in_4bit=True,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
```

Now only LoRA parameters are trainable.

---

## 6. When QLoRA is used

QLoRA is the **standard method today** for:

* Instruction tuning
* Domain adaptation (finance, legal, medical)
* Chat model personalization
* Fine-tuning open LLMs on small GPUs

Almost all modern open-source LLM fine-tuning tutorials use QLoRA.

---

## 7. Interview-level one-line explanation

**QLoRA = LoRA adapters trained on a 4-bit quantized frozen base model, enabling low-memory fine-tuning of very large LLMs.**

---

## 8. Very important intuition (most people miss this)

QLoRA does **NOT** train the quantized weights.

Instead:

* Quantized weights → frozen knowledge
* LoRA matrices → learn task-specific behavior
* During inference → both are combined

---

## 9. Extremely important interview difference

| Method      | Base weights        | Trainable      |
| ----------- | ------------------- | -------------- |
| Fine-tuning | Full precision      | All parameters |
| LoRA        | Full precision      | Small adapters |
| **QLoRA**   | **4-bit quantized** | Small adapters |

---


## Complete QLoRA Training Pipeline (Step-by-Step)

We’ll see the **real workflow used in industry**:

**Dataset → Quantized Model → LoRA injection → Training → Inference**

Libraries needed:

```bash
pip install transformers datasets peft bitsandbytes accelerate trl
```

---

# Step 1 — Load Dataset

Example instruction dataset:

```python
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train")
```

---

# Step 2 — Load Tokenizer

```python
from transformers import AutoTokenizer

model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
```

---

# Step 3 — Load Quantized Model (4-bit)

This is where **QLoRA starts**.

```python
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)
```

Now base weights are **4-bit quantized and frozen**.

---

# Step 4 — Add LoRA Adapters

```python
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
```

Now only **LoRA parameters are trainable**.

---

# Step 5 — Tokenize Dataset

```python
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize, batched=True)
```

---

# Step 6 — Training

Using HuggingFace trainer:

```python
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qlora-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()
```

---

# Step 7 — Save LoRA Adapter

Important: only adapters are saved (few MBs).

```python
model.save_pretrained("./qlora-adapter")
```

---

# Step 8 — Inference Using QLoRA Model

```python
inputs = tokenizer("Explain transformers simply:", return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0]))
```

---

# Very Important Concept (Most asked in interviews)

Training storage:

| Component    | Stored                |
| ------------ | --------------------- |
| Base LLM     | original pretrained   |
| LoRA adapter | small trained weights |

At inference:

[
Output = (Quantized\ Model) + (LoRA\ Adapter)
]

Adapters can be swapped for different tasks without retraining full model.

---

# Real-world Training Flow (Industry Standard)

1. Load pretrained LLM in **4-bit**
2. Inject LoRA
3. Train adapters on instruction dataset
4. Save adapter
5. Reuse base model + different adapters for different tasks

This is how most open-source chat models are fine-tuned today.

---

# Extremely Important Interview Insight

Why QLoRA works so well:

Large pretrained LLM already contains knowledge.
Fine-tuning mostly requires **small behavioral adjustment**, which low-rank adapters can learn efficiently.

---

## Why QLoRA Works Naturally

Now QLoRA becomes obvious:

- Base model weights **W** → quantized to **4-bit** and frozen  
- LoRA adapters **BA** → trained in higher precision  

Since adapters are small, quantizing the base model does **not** hurt learning ability.

Thus,

\[
Output = Quantized(W) + BA
\]

Memory drops dramatically, while performance remains similar.
